## **Introduction: Investigate Hurricane Helene**

**The Goal**: To investigate the impacts of Hurricane Helene by retrieving, visualizing, and exporting Multi-Radar/Multi-Sensor (MRMS) precipitation time series data. You will be able to:


* Efficiently retrieve and display precipitation time series data
* Visualize the precipitation data
* Export retrieved precipitation data to a CSV or Parquet file


**How to use this notebook:**


* **Read before running:** It is recommended that you read through the entire notebook before executing the cells.
* **Save a copy:** This notebook is a static case study. If you wish to modify the code or locations, you must save a copy to your local machine or Google Drive.
* **Navigate:** Use the Table of Contents to jump to specific sections.


**Event Context & Resources:**


* **The Event:** Hurricane Helene impacted the southeastern region heavily between September 24-27, 2024. This notebook analyzes the broader window of September 15 – September 30, 2024.
* **The Data:** Data is sourced from the NOAA MRMS dataset stored in an [Amazon Web Services (AWS) cloud bucket](https://noaa-mrms-parquet-pds.s3.us-east-1.amazonaws.com/index.html). It is verified to work with the MultiSensor_QPE and FLASH_QPE datasets.
* **Explore Further:** You can explore this data interactively using the [Precipitation Time Series Explorer (PTE)](http://ncei.noaa.gov/products/precipitation-time-series-explorer/).
* **About Us:** Created for industry-tailored resources. Visit Our Impact or contact industryproving.grounds@noaa.gov with questions.


**Notebook Details:**
* **UTC Time:** This data is shown in Coordinated Universal Time (UTC) in the example plots.
* **Automated File Save:** The files will be saved automatically to your Google Colab folder, located on the left hand side of the interface.




## **Section 1 Import Modules**

In [ ]:
# --- 1.0 Import Modules ---
import sys
import subprocess
import importlib

# Auto-install required packages if missing
required_packages = ["s3fs", "pyarrow", "polars", "matplotlib"]
missing = []
for pkg in required_packages:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"Installing missing packages: {missing}...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q"] + missing
    )
    print("Installation complete.")

# Import libraries
import s3fs
import polars as pl
import pyarrow.dataset as ds
import pyarrow.compute as pc
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MultipleLocator
from datetime import datetime, timezone
from math import radians, cos

print("Libraries imported successfully.")

---
## **Section 2: Pull in Data**
**The Goal:** To connect to the AWS cloud bucket, find the exact grid coordinates for our target locations, and efficiently download only the required data.


**How to use this section:**


**Run the cell below:** This executes a two-step optimized retrieval:
*  It quickly scans the dataset to find the exact NOAA grid coordinates closest to NCEI Asheville and Montreat (Nearest Neighbor).
*  It uses those exact coordinates to download only the precipitation data we need for our specific time window.


**Locations Assessed in this Study:**

* NCEI Asheville (35.5950, -82.5550)
* Montreat, NC (35.6450, -82.3050)


In [ ]:
# --- Section 2.0 Pull in Data ---#
# --- Configuration ---
S3_BUCKET_BASE = (
    "noaa-mrms-parquet-pds/v2/compacted/MultiSensor_QPE_01H_Pass2_00.00"
)
STATE = "North Carolina"
YEAR = 2024

# Date Window (UTC) - Naive datetime to match Parquet format
START_DT = datetime(2024, 9, 15, 0, 0)
END_DT = datetime(2024, 9, 30, 23, 59)

# Target Coordinates
ASHEVILLE_LAT, ASHEVILLE_LON = 35.5950, -82.5550
MONTREAT_LAT, MONTREAT_LON = 35.6450, -82.3050

# --- S3 Connection ---
print("Initializing S3 connection...")
fs = s3fs.S3FileSystem(anon=True)
dataset = ds.dataset(
    S3_BUCKET_BASE,
    filesystem=fs,
    format="parquet",
    partitioning="hive"
)

try:
    data_expr = (
        (pc.field("state") == STATE) &
        (pc.field("year") == YEAR) &
        (pc.field("datetime") >= START_DT) &
        (pc.field("datetime") <= END_DT) &
        (
            ((pc.field("latitude") == ASHEVILLE_LAT) &
             (pc.field("longitude") == ASHEVILLE_LON)) |
            ((pc.field("latitude") == MONTREAT_LAT) &
             (pc.field("longitude") == MONTREAT_LON))
        )
    )

    case_study_df = pl.from_arrow(dataset.to_table(filter=data_expr))
    print(
        f"Data loaded successfully. {len(case_study_df)} "
        f"{case_study_df}"
    )

except Exception as e:
    print(f"Error retrieving data: {e}")

---
## **Section 3: NCEI Asheville Plot (Single Location)**
**The Goal:** To isolate the data for our first location (NCEI Asheville) and create a time series visualization of the precipitation.


**How to use this section:**
* **Run the first cell:** This filters the targeted dataset we just downloaded for Asheville and generates a stem plot.
* **Run the second cell:** Use this cell to save your preferred outputs. You can toggle the settings to download the tabular data (CSV), the visualization (PNG), or both.


In [ ]:
# --- 3.0 Single Location Plot ---
# --- Filter & Prepare Data ---
LOCATION_NAME = "NCEI Asheville"
print(f"Preparing data for {LOCATION_NAME}...")

# 1. Filter to Asheville's specific coordinates
asheville_raw = case_study_df.filter(
    (pl.col("latitude") == ASHEVILLE_LAT) &
    (pl.col("longitude") == ASHEVILLE_LON)
).sort("datetime")

# 2. In-fill missing hourly timestamps
datetime_range_df = pl.DataFrame({
    "datetime": pl.datetime_range(
        START_DT, END_DT, interval="1h", closed="both",
        time_unit="ns", eager=True
    )
})
asheville_df = datetime_range_df.join(
    asheville_raw, on="datetime", how="left"
)

# --- Plotting (Stem Plot) ---
print("Generating plot...")
fig, ax = plt.subplots(figsize=(12, 6))

ax.stem(
    asheville_df["datetime"],
    asheville_df["observation"],
    basefmt=" ",
    label="Precipitation (in)"
)

# Formatting
ax.set_title(
    f"Precipitation: {LOCATION_NAME}\nSept 15 - Sept 30, 2024",
    fontweight='bold'
)
ax.set_ylabel("Precipitation (inches)")
ax.set_ylim(0, 2.1)
ax.yaxis.set_major_locator(MultipleLocator(0.25))
ax.yaxis.set_minor_locator(MultipleLocator(0.05))
ax.set_xlabel("Date and Time (UTC)")
start_bound = datetime(2024, 9, 15, 0, 0, tzinfo=timezone.utc)
end_bound = datetime(2024, 9, 30, 23, 59, tzinfo=timezone.utc)
ax.set_xlim(start_bound, end_bound)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax.xaxis.set_minor_locator(mdates.HourLocator(byhour=[0, 6, 12, 18]))
ax.tick_params(axis='both', which='both', direction='in')
ax.tick_params(axis='both', which='both', direction='in', top=True, right=True)
plt.xticks(rotation=45)
ax.grid(True, linestyle='--', alpha=0.7)

#Alt-text
print(
    f'A stem plot showing hourly precipitation for {LOCATION_NAME} from Sept 15 to 30, 2024.\n'
    f'The x-axis tracks time daily with 6-hour intervals, and the y-xais measures precipitation from 0 to 2.1 inches.\n'
    f'The plot highlights a significant rain event on Sept 26-27, 2024, peaking at {asheville_df["observation"].max():.2f} inches,\n'
    f'with otherwise minimal rainfall throughout the rest of the period.'
)

plt.tight_layout()
plt.show()


### **Save Options**
Files should save automatically to your Colab Folder. This can be found on the left hand side of the notebook in a folder. If you click on the folder icon, the CSV and PNG files that were saved should appear there. You can toggle the save options to False if you do not want to download any files.

In [ ]:
# --- 3.1 Save Options ---
SAVE_CSV = True
SAVE_PNG = True

if SAVE_CSV:
    csv_name = "NCEI_Asheville_Helene_2024.csv"
    asheville_csv_df = asheville_df.drop(["latitude", "longitude", "year", "state"])
    asheville_csv_df = asheville_df.rename({"observation" : "precipitation"})
    asheville_csv_df.write_csv(csv_name)
    print(f"Data saved to {csv_name}")

if SAVE_PNG:
    png_name = "NCEI_Asheville_Helene_2024.png"
    fig.savefig(png_name)
    print(f"Plot saved to {png_name}")

---
## **Section 4: Montreat, NC Plot (Multi-Location)**
**The Goal:** To expand our analysis by adding a second location (Montreat, NC) and plotting both locations simultaneously to compare intensity and timing.


**How to use this section:**
* **Run the first cell:** This filters the dataset for Montreat, NC, and combines it with the Asheville data to generate a grouped stem plot.
* **Run the second cell:** Similar to Section 3, use this cell to export the combined tabular data (CSV) and the visualization (PNG).


In [ ]:
# --- 4.0 Multi-Location Plot ---
# --- Filter Data ---
LOC2_NAME = "Montreat, NC"
print(f"Preparing data for {LOC2_NAME}...")

# 1. Filter to Montreat's specific coordinates
montreat_raw = case_study_df.filter(
    (pl.col("latitude") == MONTREAT_LAT) &
    (pl.col("longitude") == MONTREAT_LON)
).sort("datetime")

# 2. In-fill missing hourly timestamps
montreat_df = datetime_range_df.join(
    montreat_raw, on="datetime", how="left"
)

# --- Plotting (Multi-Location Stem) ---
print("Generating comparison plot...")
fig2, ax2 = plt.subplots(figsize=(12, 6))

# Plot Asheville (Using alpha for transparency so both are visible)
markerline1, stemlines1, baseline1 = ax2.stem(
    asheville_df["datetime"],
    asheville_df["observation"],
    basefmt=" ",
    label=LOCATION_NAME
)
plt.setp(stemlines1, 'color', '#0076D6', 'alpha', 0.8, 'zorder', 3)
plt.setp(
    markerline1, 'color', '#0076D6', 'marker', 'o',
    'alpha', 0.8, 'zorder', 3
)

# Plot Montreat
markerline2, stemlines2, baseline2 = ax2.stem(
    montreat_df["datetime"],
    montreat_df["observation"],
    basefmt=" ",
    label=LOC2_NAME
)
plt.setp(stemlines2, 'color', '#d95f02', 'alpha', 0.6, 'zorder', 3)
plt.setp(
    markerline2, 'color', '#d95f02', 'marker', 'x',
    'alpha', 0.6, 'zorder', 3
)

# Formatting
ax2.set_title(
    "Precipitation Comparison: Asheville vs Montreat\n"
    "Hurricane Helene (Sept 2024)",
    fontweight='bold'
)
ax2.set_ylabel("Precipitation (inches)")
ax2.set_ylim(0, 2.5)
ax2.yaxis.set_major_locator(MultipleLocator(0.25))
ax2.yaxis.set_minor_locator(MultipleLocator(0.05))
ax2.set_xlabel("Date and Time (UTC)")
start_bound = datetime(2024, 9, 15, 0, 0, tzinfo=timezone.utc)
end_bound = datetime(2024, 9, 30, 23, 59, tzinfo=timezone.utc)
ax2.set_xlim(start_bound, end_bound)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
ax2.xaxis.set_major_locator(mdates.DayLocator(interval=1))
ax2.xaxis.set_minor_locator(mdates.HourLocator(byhour=[0, 6, 12, 18]))
ax2.tick_params(axis='both', which='both', direction='in')
ax2.tick_params(axis='both', which='both', direction='in', top=True, right=True)
plt.xticks(rotation=45)

ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.7)

#Alt-text
print(
    f'A stem plot showing hourly precipitation for {LOCATION_NAME} and {LOC2_NAME} from Sept 15 to 30, 2024.\n'
    f'The x-axis tracks time daily with 6-hour intervals, and the y-xais measures precipitation from 0 to 2.5 inches.\n'
    f'The plot highlights a significant rain event on Sept 26-27, 2024, peaking at {asheville_df["observation"].max():.2f} inches in {LOCATION_NAME},\n'
    f'and {montreat_df["observation"].max():.2f} inches in {LOC2_NAME}. There is minimal rainfall throughout the rest of the period.'
)

plt.tight_layout()
plt.show()

In [ ]:
# --- 4.1 Save Options ---
if SAVE_CSV:
    # Combine for export (Outer join to ensure no timestamps are lost)
    montreat_csv_df = montreat_df.drop(["latitude", "longitude", "state", "year"])
    combined_df = asheville_csv_df.rename(
        {"precipitation": "Asheville_Precip"}
    ).join(
        montreat_csv_df.rename({"observation": "Montreat_Precip"}),
        on="datetime",
        how="full"
    ).select(["datetime", "Asheville_Precip", "Montreat_Precip"])

    multi_csv_name = "Asheville_Montreat_Comparison.csv"
    combined_df.write_csv(multi_csv_name)
    print(f"Combined data saved to {multi_csv_name}")

if SAVE_PNG:
    multi_png_name = "Asheville_Montreat_Comparison.png"
    fig2.savefig(multi_png_name)
    print(f"Comparison plot saved to {multi_png_name}")

For more information on the Precipitation Time Series Explorer product, including a more detailed notebook on the coding techniques needed for bulk data access, see the [PTE Landing Page](https://www.ncei.noaa.gov/products/precipitation-time-series-explorer/). 